In [1]:
"""
gap11_real.py
=============
Gap 11: Lifted to ℝ.

Build the algebra over ℝ with:
  Fano: same octonion-style rules (fine over ℝ)
  Identity: e_7 = 1
  TRB: t·e_i = λ·e_i, e_i·t = -λ·e_i, t·1 = λ, 1·t = t
  E: t² = 0
  G: t² = -μ  (take μ = 1)

Enumerate loops (sequences whose product is a scalar multiple of 1).
Report: count, sum of scalar values, average.

Vectorized: S[k] is an (N, N^k) array whose columns are all
left-products of length k. Loops are the scalar columns.
"""

import numpy as np
from itertools import product

N = 9
FANO = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]


def build(t_sq, lam, mu=1.0):
    """t² = t_sq; t·e_i = lam·e_i; e_i·t = -lam·e_i."""
    M = np.zeros((N, N, N), dtype=np.float64)
    for i in range(N):
        M[7, i, i] = 1.0
        M[i, 7, i] = 1.0
    for (a, b, c) in FANO:
        M[a, b, c] = 1.0
        M[b, c, a] = 1.0
        M[c, a, b] = 1.0
        M[b, a, c] = -1.0
        M[c, b, a] = -1.0
        M[a, c, b] = -1.0
    for i in range(7):
        M[i, i, 7] = -1.0
    M[8, 8, 7] = t_sq
    for i in range(7):
        M[8, i, i] = lam
        M[i, 8, i] = -lam
    M[8, 7, 7] = lam
    M[7, 8, 8] = 1.0
    return M


def left_matrices(M):
    L = np.zeros((N, N, N), dtype=np.float64)
    for i in range(N):
        for j in range(N):
            L[i, :, j] = M[i, j, :]
    return L


def is_loop_mask(S, tol=1e-9):
    """Columns of S that are scalar multiples of e_7."""
    rows_ok = np.all(np.abs(S[:7, :]) < tol, axis=0) & \
              np.all(np.abs(S[8:, :]) < tol, axis=0)
    return rows_ok


def analyze(M, name, max_L=6):
    L = left_matrices(M)
    basis = np.eye(N)

    print(f"  {name}")
    S = basis  # (N, N^1) = single-element products
    for k in range(2, max_L + 1):
        # S_new = concat([L[i] @ S for i in range(N)]) along columns
        parts = [L[i] @ S for i in range(N)]
        S = np.concatenate(parts, axis=1)

        mask = is_loop_mask(S)
        vals = S[7, mask]
        cnt = int(mask.sum())
        ssum = float(vals.sum())
        avg = ssum / cnt if cnt > 0 else 0.0

        print(f"    L={k}: loops={cnt:>12d}, sum={ssum:>18.6f}, "
              f"avg={avg:>16.6f}")
    print()


# ============================================================
# Test various λ
# ============================================================
print("=" * 72)
print("GAP 11 — ℝ-LIFTED ALGEBRA")
print("=" * 72)
print()

for lam_label, lam in [("λ=1", 1.0), ("λ=3", 3.0),
                       ("λ=11/137", 11/137), ("λ=1/137", 1/137)]:
    print(f"--- {lam_label} ---")
    print()

    M_E = build(t_sq=0.0, lam=lam)
    analyze(M_E, "E (t²=0)", max_L=6)

    M_G = build(t_sq=-1.0, lam=lam)
    analyze(M_G, "G (t²=-1)", max_L=6)

    print()

GAP 11 — ℝ-LIFTED ALGEBRA

--- λ=1 ---

  E (t²=0)
    L=2: loops=           9, sum=         -6.000000, avg=       -0.666667
    L=3: loops=          95, sum=        -20.000000, avg=       -0.210526
    L=4: loops=         925, sum=          1.000000, avg=        0.001081
    L=5: loops=        8941, sum=        190.000000, avg=        0.021250
    L=6: loops=       86055, sum=        568.000000, avg=        0.006600

  G (t²=-1)
    L=2: loops=           9, sum=         -7.000000, avg=       -0.777778
    L=3: loops=          87, sum=        -22.000000, avg=       -0.252874
    L=4: loops=         845, sum=          4.000000, avg=        0.004734
    L=5: loops=        8165, sum=        232.000000, avg=        0.028414
    L=6: loops=       78477, sum=        656.000000, avg=        0.008359


--- λ=3 ---

  E (t²=0)
    L=2: loops=           9, sum=         -6.000000, avg=       -0.666667
    L=3: loops=          95, sum=        -20.000000, avg=       -0.210526
    L=4: loops=       